# Investigation of MUSIN-G Dataset


In [64]:
from pathlib import Path

import braindecode
import mne
import scipy
import sklearn
import skorch
import torch
import torcheeg

import numpy as np
import pandas as pd
import soundfile as sf

from mne_bids import BIDSPath, find_matching_paths, get_entity_vals, read_raw_bids

In [69]:
DATA_PATH = Path("data")  # directory path of the data files for our computations
SEED = 42  # random seed for reproducibility

## Load Data

In [30]:
SONG_NAMES = {
    1: "Trip to the lonely planet",
    2: "Sail",
    3: "Concept 15",
    4: "Aurore",
    5: "Proof",
    6: "Glider",
    7: "Raag Bihag",
    8: "Albela sajan",
    9: "Mor Bani thanghat Kare",
    10: "Fallin",
    11: "Master of Running",
    12: "JB",
}

montage = mne.channels.make_standard_montage("GSN-HydroCel-129")

def LoadMUSIN_G(
    filepath: Path = DATA_PATH / "ds003774-download",
    subject_slice: slice = slice(0,20),
    sfreq: int = 125,
) -> pd.DataFrame:
    """
    Load data from https://openneuro.org/datasets/ds003774/versions/1.0.2

    Returns one row per recording (one song for one participant), with columns:
        subject, session, run, song_num, song_name, sfreq, n_times, duration_s,
        wav_path, bids_path, raw

    Note:
        - for 11 participants, the sampling rate of EEG is 1000Hz, for the remaining 9, 250Hz.
            - resample all to 125 Hz by default
        - the 1st session corresponds to song 1 (the README says each session is a song number).
        - `subject_slice` picks which participants to load.
    """
    subjects = get_entity_vals(filepath, "subject")[subject_slice]
    sessions = get_entity_vals(filepath, "session")
    task = get_entity_vals(filepath, "task")[0]
    
    rows = []
    for sub in subjects:
        print("Subject: " + sub)
        for ses in sessions:
            print("Session: " + ses)
            paths = find_matching_paths(
                subjects=sub,
                sessions=ses,
                tasks=task,
                extensions=[".set"],
                suffixes="eeg",
                root=filepath,
            )
            song_num = int(ses)
            song_file = filepath / "Code" / "ESongs" / f"{song_num}.esh.wav"

            for path in paths:
                raw = read_raw_bids(path)
                raw.resample(sfreq = sfreq)
                
                # Provided data names the 'Cz" electrode, "E129"
                raw.rename_channels({'E129': 'Cz'})
                raw.set_montage(montage)
    
                audio, audio_rate = sf.read(song_file)
                
                raw.load_data()
    
                rows.append(
                    {
                        "subject": sub,
                        "session": ses,
                        "run": path.run,
                        "song_num": song_num,
                        "song_name": SONG_NAMES[song_num],
                        "wav_path": song_file,
                        "audio": audio,
                        "audio_rate": audio_rate,
                        "bids_path": path,
                        "raw": raw,
                    }
                )

    return pd.DataFrame(rows)

In [31]:
df = LoadMUSIN_G(subject_slice = slice(0, 5))

Subject: 001
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 002
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 003
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 004
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 005
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 006
Session: 01
Session: 02
Session: 03
Session: 04
Session: 05
Session: 06
Session: 07
Session: 08
Session: 09
Session: 10
Session: 11
Session: 12
Subject: 007
Session: 01
Session: 02
Session: 03
Session: 

In [ ]:
df

In [ ]:
df.memory_usage()

## Preprocess data


In [33]:
# Cz is the reference electrode, the data was already referenced to Cz and is flat
# Exclude it from model inputs
df["raw"].apply(lambda x: x.drop_channels("Cz") if "Cz" in x.ch_names else x)

# Cut the 10s of silence from the raw recording
df["raw"].apply(lambda x: x.copy().crop(tmin=10.0))


0      <RawEEGLAB | sub-001_ses-01_task-MusicListenin...
1      <RawEEGLAB | sub-001_ses-02_task-MusicListenin...
2      <RawEEGLAB | sub-001_ses-03_task-MusicListenin...
3      <RawEEGLAB | sub-001_ses-04_task-MusicListenin...
4      <RawEEGLAB | sub-001_ses-05_task-MusicListenin...
                             ...                        
115    <RawEEGLAB | sub-010_ses-08_task-MusicListenin...
116    <RawEEGLAB | sub-010_ses-09_task-MusicListenin...
117    <RawEEGLAB | sub-010_ses-10_task-MusicListenin...
118    <RawEEGLAB | sub-010_ses-11_task-MusicListenin...
119    <RawEEGLAB | sub-010_ses-12_task-MusicListenin...
Name: raw, Length: 120, dtype: object

## Classify the song based on EEG

In [40]:
type(df["raw"])

pandas.Series

In [51]:
from sklearn.model_selection import GroupShuffleSplit

epoch, song, subject = [], [], []
for index, row in df.iterrows():
    print("Index:", index)
    ep = mne.make_fixed_length_epochs(
        row["raw"].copy(), duration=2, overlap=1
    )
    epoch.append(ep.get_data())
    song.append([row["song_name"]] * len(ep))
    subject.append([row["subject"]] * len(ep))
    
epoch = np.concatenate(epoch)
song = np.concatenate(song)
subject = np.concatenate(subject)


Index: 0
Index: 1
Index: 2
Index: 3
Index: 4
Index: 5
Index: 6
Index: 7
Index: 8
Index: 9
Index: 10
Index: 11
Index: 12
Index: 13
Index: 14
Index: 15
Index: 16
Index: 17
Index: 18
Index: 19
Index: 20
Index: 21
Index: 22
Index: 23
Index: 24
Index: 25
Index: 26
Index: 27
Index: 28
Index: 29
Index: 30
Index: 31
Index: 32
Index: 33
Index: 34
Index: 35
Index: 36
Index: 37
Index: 38
Index: 39
Index: 40
Index: 41
Index: 42
Index: 43
Index: 44
Index: 45
Index: 46
Index: 47
Index: 48
Index: 49
Index: 50
Index: 51
Index: 52
Index: 53
Index: 54
Index: 55
Index: 56
Index: 57
Index: 58
Index: 59
Index: 60
Index: 61
Index: 62
Index: 63
Index: 64
Index: 65
Index: 66
Index: 67
Index: 68
Index: 69
Index: 70
Index: 71
Index: 72
Index: 73
Index: 74
Index: 75
Index: 76
Index: 77
Index: 78
Index: 79
Index: 80
Index: 81
Index: 82
Index: 83
Index: 84
Index: 85
Index: 86
Index: 87
Index: 88
Index: 89
Index: 90
Index: 91
Index: 92
Index: 93
Index: 94
Index: 95
Index: 96
Index: 97
Index: 98
Index: 99
Index: 100

In [61]:
gss = GroupShuffleSplit(n_splits=1, test_size=2, random_state=SEED)
train_idx, test_idx = next(gss.split(epoch, song, groups=subject))

Xtrain, Xtest = epoch[train_idx], epoch[test_idx]
ytrain, ytest = song[train_idx], song[test_idx]

## Train the Model

In [62]:
from braindecode.models import EEGNet
from braindecode.datautil import infer_signal_properties

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(song)
y_enc = le.transform(song)

ytrain_enc = le.transform(ytrain)
ytest_enc = le.transform(ytest)

sig_props = infer_signal_properties(epoch, song, mode="classification")

eegnet_model = EEGNet(
    n_chans=sig_props["n_chans"],
    n_outputs=sig_props["n_outputs"],
    n_times=sig_props["n_times"],
    final_conv_length="auto",
    F1=8,
    D=2,
)



In [73]:
from braindecode import EEGClassifier
from skorch.dataset import ValidSplit

cuda = torch.cuda.is_available()
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True

clf = EEGClassifier(
    eegnet_model,
    criterion=torch.nn.CrossEntropyLoss,
    optimizer=torch.optim.AdamW,
    train_split=ValidSplit(0.2, random_state=SEED),
    device=device,
    classes=[SONG_NAMES[i] for i in range(1, 13)]
)


In [74]:
clf.fit(Xtrain, ytrain_enc)

  epoch    train_loss    valid_acc    valid_loss      dur
-------  ------------  -----------  ------------  -------
      1        2.3504       0.1134        2.5797  61.1070
      2        1.9222       0.1297        2.5514  59.7638
      3        1.6366       0.1862        2.8122  59.9811
      4        1.4907       0.1947        2.8685  58.2953
      5        1.3315       0.2451        2.5208  63.2682
      6        1.2601       0.2419        2.9531  63.8512
      7        1.1787       0.2472        3.3217  60.3194
      8        1.0729       0.3508        2.1045  63.6445
      9        1.0290       0.4561        1.7261  58.7650
     10        0.9062       0.4537        1.6840  59.3777


,module,EEGNet( (en...') ) ) )
,optimizer,<class 'torch....adamw.AdamW'>
,lr,0.01
,max_epochs,10
,batch_size,128
,iterator_train,<class 'torch...r.DataLoader'>
,iterator_valid,<class 'torch...r.DataLoader'>
,dataset,<class 'skorc...aset.Dataset'>
,train_split,<skorch.datas...x7f12addabad0>
,predict_nonlinearity,'auto'
,warm_start,False


In [77]:
from braindecode.visualization import plot_confusion_matrix
from sklearn.metrics import confusion_matrix, accuracy_score

score = accuracy_score(ytest_enc, clf.predict(Xtest))
print("Score:", score)

score = accuracy_score(ytrain_enc, clf.predict(Xtrain))
print("Score:", score)

Score: 0.14606376057254392
Score: 0.44461613532856215
